# ZuCo 2.0 EEG Data Extraction & Visualization Pipeline
This notebook provides a pure Python pipeline to read ZuCo 2.0 `.mat` files (v7.3 format), extract the EEG data, and visualize the EEG mappings at both the **sentence level** and **word level**.

### Prerequisites
Make sure you have the required libraries installed:
```bash
pip install h5py numpy matplotlib scipy
```

### Download ZuCo 2.0 Dataset
The following cell contains the download script to automatically download the ZuCo 2.0 dataset from OSF (approx 120GB). It supports resuming if interrupted.

In [ ]:
import os
import sys

try:
    import requests
    from tqdm import tqdm
except ImportError:
    print("Please install required packages before running:")
    print("pip install requests tqdm")
    sys.exit(1)

# OSF Node ID for ZuCo 2.0
NODE_ID = "2urht"
BASE_URL = f"https://api.osf.io/v2/nodes/{NODE_ID}/files/osfstorage/"

# Resolve dataset directory (../dataset/zuco2)
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")

import time


def get_osf_data(api_url, max_retries=5):
    """Fetch folder metadata from OSF API with retries, handling pagination."""
    all_data = []
    current_url = api_url

    while current_url:
        for attempt in range(max_retries):
            try:
                response = requests.get(current_url, timeout=30)
                response.raise_for_status()
                json_data = response.json()
                all_data.extend(json_data["data"])

                # Check for next page
                links = json_data.get("links", {})
                current_url = links.get("next")
                break  # Break retry loop if successful
            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    print(
                        f"Network error while fetching metadata: {e}. Retrying in 5s... ({attempt + 1}/{max_retries})"
                    )
                    time.sleep(5)
                else:
                    raise
    return all_data


def download_file_resumable(url, destination, max_retries=5):
    """Downloads a file with resume support and retries."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temp_destination = destination + ".tmp"

    for attempt in range(max_retries):
        file_size = 0
        if os.path.exists(destination):
            print(
                f"File {os.path.basename(destination)} already fully downloaded. Skipping."
            )
            return True

        if os.path.exists(temp_destination):
            file_size = os.path.getsize(temp_destination)

        headers = {"Range": f"bytes={file_size}-"} if file_size > 0 else {}

        try:
            response = requests.get(
                url, headers=headers, stream=True, allow_redirects=True, timeout=30
            )

            # 416 Range Not Satisfiable means we requested a range past the end of the file
            if response.status_code == 416:
                os.rename(temp_destination, destination)
                print(f"File {os.path.basename(destination)} already fully downloaded.")
                return True

            if response.status_code not in [200, 206]:
                print(f"Failed to download {url}. Status code: {response.status_code}")
                return False

            if file_size > 0 and response.status_code == 200:
                print("Server doesn't support resume. Restarting download...")
                file_size = 0
                mode = "wb"
            else:
                mode = "ab"

            total_size = int(response.headers.get("content-length", 0)) + file_size

            with (
                open(temp_destination, mode) as f,
                tqdm(
                    desc=os.path.basename(destination),
                    total=total_size,
                    initial=file_size,
                    unit="iB",
                    unit_scale=True,
                    unit_divisor=1024,
                ) as bar,
            ):
                for chunk in response.iter_content(chunk_size=8192 * 4):
                    if chunk:
                        size = f.write(chunk)
                        bar.update(size)

            # Verify the file is complete
            if total_size == 0 or os.path.getsize(temp_destination) >= total_size:
                os.rename(temp_destination, destination)
                return True
            else:
                print(f"Download incomplete for {os.path.basename(destination)}")
                # Will retry in the next loop iteration

        except (OSError, KeyError, ValueError) as e:
            if attempt < max_retries - 1:
                print(
                    f"\nError downloading {os.path.basename(destination)}: {e}. Retrying in 5s... ({attempt + 1}/{max_retries})"
                )
                time.sleep(5)
            else:
                print(
                    f"\nFailed to download {os.path.basename(destination)} after {max_retries} attempts: {e}"
                )
                return False


def traverse_and_download(api_url, current_path):
    """Recursively traverses OSF folders and downloads files sequentially."""
    print(f"Fetching listing for {os.path.relpath(current_path, PROJECT_ROOT)} ...")
    items = get_osf_data(api_url)

    for item in items:
        kind = item["attributes"]["kind"]
        name = item["attributes"]["name"]

        if kind == "folder":
            next_url = item["relationships"]["files"]["links"]["related"]["href"]
            next_path = os.path.join(current_path, name)
            success = traverse_and_download(next_url, next_path)
            if not success:
                return False
        elif kind == "file":
            download_url = item["links"]["download"]
            file_path = os.path.join(current_path, name)

            # Download file sequentially, stopping if one fails
            success = download_file_resumable(download_url, file_path)
            if not success:
                print(f"Stopping download process because {name} failed.")
                return False

    return True


if __name__ == "__main__":
    print(f"Starting download of ZuCo 2.0 to: {DATASET_DIR}")
    print("-" * 50)
    os.makedirs(DATASET_DIR, exist_ok=True)

    success = traverse_and_download(BASE_URL, DATASET_DIR)

    if success:
        print("-" * 50)
        print("Dataset download completed successfully!")
    else:
        print("-" * 50)
        print("Dataset download interrupted. Run the script again to resume.")

In [ ]:
import glob
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np

# Setup paths
PROJECT_ROOT = os.path.dirname(os.getcwd())
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")

# Output directories for visualizations
WORD_MAPPING_DIR = os.path.join(PROJECT_ROOT, "dataset", "word eeg mapping")
SENTENCE_MAPPING_DIR = os.path.join(PROJECT_ROOT, "dataset", "sentence eeg mapping")

os.makedirs(WORD_MAPPING_DIR, exist_ok=True)
os.makedirs(SENTENCE_MAPPING_DIR, exist_ok=True)

print(f"Data directory: {DATASET_DIR}")
print(f"Word mapping output: {WORD_MAPPING_DIR}")
print(f"Sentence mapping output: {SENTENCE_MAPPING_DIR}")

### Helper Functions
Functions to extract strings from `h5py` references and plot the EEG graphs.

In [ ]:
def get_string(f, ref):
    """Extracts string from h5py object reference."""
    try:
        obj = f[ref]
        return "".join(chr(c[0]) for c in obj[:])
    except Exception:  # noqa: BLE001
        return "Unknown"


CHANNEL_LABELS = [
    "E2",
    "E3",
    "E4",
    "E5",
    "E6",
    "E7",
    "E9",
    "E10",
    "E11",
    "E12",
    "E13",
    "E15",
    "E16",
    "E18",
    "E19",
    "E20",
    "E22",
    "E23",
    "E24",
    "E26",
    "E27",
    "E28",
    "E29",
    "E30",
    "E31",
    "E33",
    "E34",
    "E35",
    "E36",
    "E37",
    "E38",
    "E39",
    "E40",
    "E41",
    "E42",
    "E43",
    "E44",
    "E45",
    "E46",
    "E47",
    "E50",
    "E51",
    "E52",
    "E53",
    "E54",
    "E55",
    "E57",
    "E58",
    "E59",
    "E60",
    "E61",
    "E62",
    "E64",
    "E65",
    "E66",
    "E67",
    "E69",
    "E70",
    "E71",
    "E72",
    "E74",
    "E75",
    "E76",
    "E77",
    "E78",
    "E79",
    "E80",
    "E82",
    "E83",
    "E84",
    "E85",
    "E86",
    "E87",
    "E89",
    "E90",
    "E91",
    "E92",
    "E93",
    "E95",
    "E96",
    "E97",
    "E98",
    "E100",
    "E101",
    "E102",
    "E103",
    "E104",
    "E105",
    "E106",
    "E108",
    "E109",
    "E110",
    "E111",
    "E112",
    "E114",
    "E115",
    "E116",
    "E117",
    "E118",
    "E120",
    "E121",
    "E122",
    "E123",
    "E124",
    "Cz",
]


def plot_eeg(eeg_data, title, filename, channel_labels=CHANNEL_LABELS):
    """
    Plots all EEG channels on a single heatmap graph and saves the image.
    eeg_data shape is usually (channels, time) or (time, channels).
    """
    if eeg_data is None or eeg_data.size == 0:
        return

    eeg_data = np.array(eeg_data)
    # Ensure shape is (channels, time)
    if eeg_data.shape[0] > eeg_data.shape[1]:
        eeg_data = eeg_data.T

    num_channels, _time_points = eeg_data.shape

    fig, ax = plt.subplots(figsize=(15, 20))
    cax = ax.imshow(eeg_data, aspect="auto", cmap="viridis")
    fig.colorbar(cax, ax=ax, label="Amplitude")

    ax.set_title(title)
    ax.set_xlabel("Time points")
    ax.set_ylabel("EEG Channels")

    if channel_labels and num_channels == len(channel_labels):
        ax.set_yticks(np.arange(num_channels))
        ax.set_yticklabels(channel_labels, fontsize=8)
    else:
        ax.set_yticks(np.arange(0, num_channels, max(1, num_channels // 20)))

    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()

### Extraction and Visualization Loop
This loop iterates through the downloaded `.mat` files, extracts sentence and word data, and plots them.

In [ ]:
# Find all .mat files in the dataset directory
mat_files = glob.glob(os.path.join(DATASET_DIR, "**", "results*.mat"), recursive=True)
print(f"Found {len(mat_files)} .mat files.")

if not mat_files:
    print("No .mat files found. Make sure the download script has finished running.")

# Loop through files
for mat_file in mat_files:  # Let's test just the first 3 files
    subject_name = os.path.basename(mat_file).replace(".mat", "")
    print(f"Processing {subject_name}...")

    try:
        with h5py.File(mat_file, "r") as f:
            if "sentenceData" not in f:
                continue

            sd = f["sentenceData"]
            num_sentences = sd["content"].shape[0]

            # Limit to first 2 sentences for testing
            for i in range(num_sentences):
                # 1. Extract Sentence
                content_ref = sd["content"][i, 0]
                sent_text = get_string(f, content_ref)
                safe_sent_name = "".join(
                    [c if c.isalnum() else "_" for c in sent_text]
                )[:50]

                # 2. Sentence EEG Mapping
                eeg_sent = None
                for key in ["rawData", "mean_t1", "mean_t2"]:
                    if key in sd:
                        ref = sd[key][i, 0]
                        if ref:
                            eeg_sent = f[ref][:]
                        break

                if eeg_sent is not None:
                    out_name = os.path.join(
                        SENTENCE_MAPPING_DIR,
                        f"{subject_name}_sent_{i}_{safe_sent_name}.png",
                    )
                    plot_eeg(eeg_sent, f"Sentence EEG: {sent_text}", out_name)

                # 3. Word EEG Mapping
                if "word" in sd:
                    word_refs = sd["word"][i, 0]
                    if word_refs:
                        words = f[word_refs]
                        num_words = words["content"].shape[0]

                        # Limit to first 3 words for testing
                        for w in range(num_words):
                            w_ref = words["content"][w, 0]
                            w_text = get_string(f, w_ref)
                            safe_word_name = "".join(
                                [c if c.isalnum() else "_" for c in w_text]
                            )

                            eeg_word = None
                            for key in ["rawEEG", "FFD_t1", "TRT_t1"]:
                                if key in words:
                                    e_ref = words[key][w, 0]
                                    if e_ref:
                                        eeg_word = f[e_ref][:]
                                    break

                            if eeg_word is not None:
                                out_name = os.path.join(
                                    WORD_MAPPING_DIR,
                                    f"{subject_name}_sent_{i}_word_{w}_{safe_word_name}.png",
                                )
                                plot_eeg(eeg_word, f"Word EEG: {w_text}", out_name)

        print(f"Finished processing {subject_name}.")
    except (OSError, KeyError, ValueError) as e:
        print(f"Skipping {mat_file}: not a valid v7.3 HDF5 or no sentenceData ({e})")

print("Visualization pipeline completed. Check the 'dataset' subfolders for images.")